# 02 — Pneumonia Detection: CNN Models

Train and evaluate three baseline CNN architectures on the chest X-ray dataset:

| Model | Description |
|-------|-------------|
| **A** | Baseline CNN without data augmentation |
| **B** | Baseline CNN with in-graph data augmentation |
| **C** | Baseline CNN with Keras Tuner hyperparameter optimisation |

All models use sklearn-balanced class weighting via `compute_class_weights()`, decision-threshold tuning on PNEUMONIA F1, and evaluation on a held-out test set.

**Sections**
1. Environment Setup
2. Configuration
3. Data Loading & Preprocessing
4. Model A — Baseline CNN (No Augmentation)
5. Model B — Baseline CNN (With Augmentation)
6. Overfitting Countermeasures
7. Model C — Baseline CNN (Tuned Hyperparameters)
8. Conclusion — CNN Model Comparison

## 1. Environment Setup

Import all dependencies, install optional packages, and set random seeds.

In [ ]:
import os
import sys
import json
import random
import pathlib
import subprocess

# Install optional dependencies if missing
for _pkg in ["keras-tuner"]:
    try:
        __import__(_pkg.replace("-", "_"))
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], check=False)

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

import tensorflow as tf
import keras
import keras_tuner as kt

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

KAGGLE_HELPERS_PARENT = "/kaggle/input/datasets/omosaad/helpers"
KAGGLE_HELPERS_DIR = "/kaggle/input/datasets/omosaad/helpers/helpers"

if "/content/drive/MyDrive/Colab Notebooks" not in sys.path:
    sys.path.append("/content/drive/MyDrive/Colab Notebooks")

try:
    from helpers import data_utils, model_utils, training_utils, visualization
except ModuleNotFoundError:
    for _p in [KAGGLE_HELPERS_PARENT, KAGGLE_HELPERS_DIR]:
        if os.path.isdir(_p) and _p not in sys.path:
            sys.path.append(_p)
    from helpers import data_utils, model_utils, training_utils, visualization

keras.utils.set_random_seed(42)
np.random.seed(42)
random.seed(42)

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)

## 2. Configuration

Dataset paths, image size, batch size, checkpoint locations, and runtime flags.

In [ ]:
USE_COLAB = True
USE_KAGGLE = False  # Set True when running on Kaggle

In [ ]:
# Configuration
if USE_KAGGLE:
    DATASET_ROOT = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray/"
    SAVE_DIR = pathlib.Path("/kaggle/working/saved_models")
elif USE_COLAB:
    from google.colab import drive

    drive.mount('/content/drive')
    DATASET_ROOT = "/content/drive/MyDrive/x-ray-dataset/"
    SAVE_DIR = pathlib.Path("/content/drive/MyDrive/saved_models")
else:
    DATASET_ROOT = "dataset"
    SAVE_DIR = pathlib.Path("saved_models")

TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
VAL_DIR = os.path.join(DATASET_ROOT, "val")  # merged into training pool if present
TEST_DIR = os.path.join(DATASET_ROOT, "test")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20
VAL_SPLIT = 0.1
AUTOTUNE = tf.data.AUTOTUNE

SAVE_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_WITHOUT_AUG_CHECKPOINT_PATH = SAVE_DIR / "baseline_without_aug_best_checkpoint.keras"
BASELINE_CHECKPOINT_PATH = SAVE_DIR / "baseline_best_checkpoint.keras"
TUNED_CHECKPOINT_PATH = SAVE_DIR / "baseline_tuned_best_checkpoint.keras"

for split_path in [TRAIN_DIR, TEST_DIR]:
    if not os.path.isdir(split_path):
        raise FileNotFoundError(f"Missing directory: {split_path}")

runtime_name = "Kaggle" if USE_KAGGLE else ("Google Colab" if USE_COLAB else "Local")
print(f"Environment: {runtime_name}")
print(f"Dataset root: {DATASET_ROOT}")
print(f"Model artifacts will be stored in: {SAVE_DIR}")

## 3. Data Loading & Preprocessing

Build stratified train/validation splits from the training pool, construct balanced `tf.data` pipelines, and define the augmentation stack.

In [ ]:
# Collect all file paths and labels from train plus optional original val directory
all_train_paths, all_train_labels, test_paths, test_labels = data_utils.gather_split_paths_labels(
    TRAIN_DIR,
    TEST_DIR,
    VAL_DIR,
)

print(f"Source pool size (train + optional original val): {len(all_train_paths)} images")

In [ ]:
# Build train/val/test datasets from helper
train_ds, val_ds, test_ds, dataset_meta = data_utils.build_train_val_test_datasets(
    train_paths=all_train_paths,
    train_labels=all_train_labels,
    test_paths=test_paths,
    test_labels=test_labels,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    val_split=VAL_SPLIT,
    seed=42,
    preprocess_fn=None,
    autotune=AUTOTUNE,
)

train_paths = dataset_meta["train_paths"]
train_labels_raw = dataset_meta["train_labels_raw"]
val_paths = dataset_meta["val_paths"]
val_labels_arr = dataset_meta["val_labels"]
normal_count = dataset_meta["normal_count"]
pneumonia_count = dataset_meta["pneumonia_count"]
steps_per_epoch = dataset_meta["steps_per_epoch"]

print(f"Stratified split (before balancing): {len(train_paths)} train, {len(val_paths)} val")
print(f"  Train before -> NORMAL={normal_count}, PNEUMONIA={pneumonia_count}")
print(f"  Val          -> NORMAL={int((val_labels_arr==0).sum())}, PNEUMONIA={int((val_labels_arr==1).sum())}")
print(f"  Balanced sampling -> NORMAL:PNEUMONIA = 1:1 per epoch")
print(f"  Steps per epoch: {steps_per_epoch}")

class_names = ["NORMAL", "PNEUMONIA"]
print("Class names:", class_names)
print("Note: dataset/val folder excluded; validation split is stratified from dataset/train.")

In [ ]:
class_weight = data_utils.compute_class_weights(train_labels_raw, strategy="balanced")
print("Computed class weights:", class_weight)

In [ ]:
# Augmentation layers
data_augmentation = data_utils.build_data_augmentation(
    rotation=30,
    width_shift=0.1,
    height_shift=0.1,
    shear=0.2,
    zoom=0.2,
)

In [ ]:
# Performance optimization for tf.data pipelines
# train_ds is infinite (repeat + sample_from_datasets), so we only prefetch.
train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

print("Datasets are ready.")

### 3.1 Balanced Sampling & Augmentation Verification

Confirm that the balanced sampling strategy and augmentation pipeline produce the expected output.

In [ ]:
normal_paths = train_paths[train_labels_raw == 0]
pneumonia_paths = train_paths[train_labels_raw == 1]

# For diagnostics only: effective per-epoch class counts under 1:1 sampled training.
balanced_epoch_count = steps_per_epoch * BATCH_SIZE // 2

print("Class distribution check")
print(f"  Train before balancing: NORMAL={(train_labels_raw==0).sum()}, PNEUMONIA={(train_labels_raw==1).sum()}")
print(f"  Train after balancing : NORMAL~{balanced_epoch_count}, PNEUMONIA~{balanced_epoch_count} (sampled)")
print(f"  Validation            : NORMAL={(val_labels_arr==0).sum()}, PNEUMONIA={(val_labels_arr==1).sum()}")

rng = np.random.default_rng(42)
fig, axes = plt.subplots(2, 8, figsize=(18, 5))
normal_sample_paths = rng.choice(normal_paths, size=min(8, len(normal_paths)), replace=False)
pneumonia_sample_paths = rng.choice(pneumonia_paths, size=min(8, len(pneumonia_paths)), replace=False)

for i in range(8):
    if i < len(normal_sample_paths):
        img = keras.utils.load_img(normal_sample_paths[i], target_size=IMG_SIZE)
        axes[0, i].imshow(img)
        axes[0, i].set_title("NORMAL")
    axes[0, i].axis("off")

    if i < len(pneumonia_sample_paths):
        img = keras.utils.load_img(pneumonia_sample_paths[i], target_size=IMG_SIZE)
        axes[1, i].imshow(img)
        axes[1, i].set_title("PNEUMONIA")
    axes[1, i].axis("off")

plt.suptitle("Original Training Samples (before augmentation)")
plt.tight_layout()
plt.show()

sample_normal = keras.utils.load_img(normal_paths[0], target_size=IMG_SIZE)
sample_tensor = tf.expand_dims(tf.cast(keras.utils.img_to_array(sample_normal), tf.float32), axis=0)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for idx, ax in enumerate(axes.flat):
    aug_img = data_augmentation(sample_tensor, training=True)[0]
    aug_img = tf.clip_by_value(aug_img / 255.0, 0.0, 1.0)
    ax.imshow(aug_img)
    ax.set_title(f"Augmented #{idx+1}")
    ax.axis("off")

plt.suptitle("One NORMAL image with 8 random augmentations")
plt.tight_layout()
plt.show()

### 3.2 Helper Utilities

Bind the shared model-saver closure to this notebook’s save directory.

In [ ]:
# Bind a notebook-local saver without redefining utility functions here.
save_model_with_meta = training_utils.make_model_saver(SAVE_DIR)

## 4. Model A — Baseline CNN (No Augmentation)

A regularised CNN with batch normalisation, progressive dropout, and L2 regularisation — trained **without** any data augmentation to serve as a lower-bound reference.

In [ ]:
# Baseline CNN model (no transfer-learning backbone)
model_without_aug = model_utils.build_baseline_cnn(
    img_size=IMG_SIZE,
    augmentation_layer=None,
    learning_rate=1e-3,
    l2_reg=1e-4,
    name="baseline_pneumonia_cnn",
)
model_without_aug.summary()

### 4.1 Training

In [ ]:
my_callbacks_no_aug = training_utils.get_training_callbacks(
    checkpoint_path=BASELINE_WITHOUT_AUG_CHECKPOINT_PATH,
    patience=5,
    monitor="val_auc",
)

history_model_without_aug = model_without_aug.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    callbacks=my_callbacks_no_aug,  
    class_weight=class_weight,
)

### 4.2 Evaluation

In [ ]:
# Plot training curves (loss, accuracy, and AUC)
if history_model_without_aug is None:
    print("Training history is unavailable because model was loaded from checkpoint.")
else:
    plt.figure(figsize=(15, 4))

    plt.subplot(1, 3, 1)
    plt.plot(history_model_without_aug.history["accuracy"], label="Train Accuracy")
    plt.plot(history_model_without_aug.history["val_accuracy"], label="Val Accuracy")
    plt.title("Accuracy Over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()

    plt.subplot(1, 3, 2)
    plt.plot(history_model_without_aug.history["loss"], label="Train Loss")
    plt.plot(history_model_without_aug.history["val_loss"], label="Val Loss")
    plt.title("Loss Over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    plt.subplot(1, 3, 3)
    plt.plot(history_model_without_aug.history["auc"], label="Train AUC")
    plt.plot(history_model_without_aug.history["val_auc"], label="Val AUC")
    plt.title("AUC Over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("AUC")
    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
best_model_without_aug = training_utils.load_model_compat(BASELINE_WITHOUT_AUG_CHECKPOINT_PATH)
print(f"Loaded best model from: {BASELINE_WITHOUT_AUG_CHECKPOINT_PATH}")

In [ ]:
# Threshold selection stage
print("Searching the best decision threshold on validation data (macro F1)...")

In [ ]:
# Threshold tuning on validation split (PNEUMONIA F1)
best_threshold_no_aug, best_f1_no_aug, _, _ = training_utils.tune_threshold(
    best_model_without_aug, val_ds
)
print(f"Best threshold (PNEUMONIA F1): {best_threshold_no_aug:.2f}  |  Best F1: {best_f1_no_aug:.4f}")

In [ ]:
# Evaluate on test set
metrics_no_aug, report_no_aug, y_true, y_prob, y_pred = training_utils.evaluate_model(
    best_model_without_aug, test_ds, best_threshold_no_aug
)
test_roc_auc = metrics_no_aug["auc"]
print(f"Test ROC-AUC: {test_roc_auc:.4f}")

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=["NORMAL", "PNEUMONIA"],
    yticklabels=["NORMAL", "PNEUMONIA"],
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix – No Aug (threshold={best_threshold_no_aug:.2f})")
plt.tight_layout()
plt.show()

print("Classification Report (Test):\n")
print(classification_report(y_true, y_pred, target_names=["NORMAL", "PNEUMONIA"], zero_division=0))

### 4.3 Sample Predictions

In [ ]:
# Show sample predictions from test set (balanced sample: 5 NORMAL + 4 PNEUMONIA)
normal_images, pneumonia_images = [], []
normal_true, pneumonia_true = [], []

for images, labels in test_ds:
    lbls = labels.numpy().ravel().astype(int)
    for i, lbl in enumerate(lbls):
        if lbl == 0 and len(normal_images) < 5:
            normal_images.append(images[i])
            normal_true.append(lbl)
        elif lbl == 1 and len(pneumonia_images) < 4:
            pneumonia_images.append(images[i])
            pneumonia_true.append(lbl)

    if len(normal_images) >= 5 and len(pneumonia_images) >= 4:
        break

sample_images = tf.stack(normal_images + pneumonia_images)
sample_true = np.array(normal_true + pneumonia_true, dtype=int)

sample_probs = best_model_without_aug.predict(sample_images, verbose=0).ravel()
sample_preds = (sample_probs >= best_threshold).astype(int)

idx_to_class = {0: "NORMAL", 1: "PNEUMONIA"}

plt.figure(figsize=(12, 12))
plt.suptitle(f"Sample Test Predictions (threshold={best_threshold:.2f})", y=1.02)
for i in range(len(sample_images)):
    img = sample_images[i].numpy().astype("uint8")
    true_label = idx_to_class[int(sample_true[i])]
    pred_label = idx_to_class[int(sample_preds[i])]
    conf = sample_probs[i] if sample_preds[i] == 1 else (1 - sample_probs[i])

    plt.subplot(3, 3, i + 1)
    plt.imshow(img.astype("uint8"))
    plt.title(f"T:{true_label} | P:{pred_label}\nConf:{conf:.2f}")
    plt.axis("off")

plt.tight_layout()
plt.show()

### 4.4 Save Model Artifacts

In [ ]:
# Save Model A artifacts
no_aug_hyperparams = {
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "optimizer": "Adam",
    "learning_rate": 1e-3,
    "augmentation": False,
    "class_weight": class_weight,
    "steps_per_epoch": steps_per_epoch,
}

save_model_with_meta(
    best_model_without_aug, "baseline_cnn_no_aug",
    metrics_no_aug, history_model_without_aug,
    no_aug_hyperparams, best_threshold_no_aug,
)
print(f"Model A saved with threshold={best_threshold_no_aug:.2f}")

## 5. Model B — Baseline CNN (With Augmentation)

The same CNN architecture as Model A, but with an in-graph augmentation layer (`RandomFlip`, `RandomRotation`, `RandomTranslation`, `RandomZoom`, `RandomShear`, `RandomBrightness`).  The augmentation layer is part of the model and is only active during training.

In [ ]:
# Baseline CNN model (no transfer-learning backbone)
model = model_utils.build_baseline_cnn(
    img_size=IMG_SIZE,
    augmentation_layer=data_augmentation,
    learning_rate=1e-3,
    l2_reg=1e-4,
    name="pneumonia_cnn",
)
model.summary()

### 5.1 Dataset Summary

In [ ]:
print("Balanced training split distribution (sampled):")
print(f"  NORMAL~{steps_per_epoch * BATCH_SIZE // 2}, PNEUMONIA~{steps_per_epoch * BATCH_SIZE // 2}, TOTAL~{steps_per_epoch * BATCH_SIZE}")
print("Validation split distribution:")
print(f"  NORMAL={int((val_labels_arr==0).sum())}, PNEUMONIA={int((val_labels_arr==1).sum())}, TOTAL={len(val_labels_arr)}")

print("Model is already compiled by model_utils.build_baseline_cnn with PR-AUC and gradient clipping.")

### 5.2 Training

In [ ]:
my_callbacks = training_utils.get_training_callbacks(
    checkpoint_path=BASELINE_CHECKPOINT_PATH,
    patience=5,
    monitor="val_auc",
)

if BASELINE_CHECKPOINT_PATH.exists():
    print(f"Loading existing baseline checkpoint: {BASELINE_CHECKPOINT_PATH}")
    model = training_utils.load_model_compat(BASELINE_CHECKPOINT_PATH)
    history = None
else:
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        steps_per_epoch=steps_per_epoch,
        callbacks=my_callbacks,
        class_weight=class_weight,
    )
    print(f"Best baseline checkpoint saved to: {BASELINE_CHECKPOINT_PATH}")

### 5.3 Evaluation

In [ ]:
# Plot training curves (loss, accuracy, and AUC)
if history is None:
    print("Training history is unavailable because model was loaded from checkpoint.")
else:
    plt.figure(figsize=(15, 4))

    plt.subplot(1, 3, 1)
    plt.plot(history.history["accuracy"], label="Train Accuracy")
    plt.plot(history.history["val_accuracy"], label="Val Accuracy")
    plt.title("Accuracy Over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()

    plt.subplot(1, 3, 2)
    plt.plot(history.history["loss"], label="Train Loss")
    plt.plot(history.history["val_loss"], label="Val Loss")
    plt.title("Loss Over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    plt.subplot(1, 3, 3)
    plt.plot(history.history["auc"], label="Train AUC")
    plt.plot(history.history["val_auc"], label="Val AUC")
    plt.title("AUC Over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("AUC")
    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# Baseline checkpoint loading stage
print("Loading baseline checkpoint for threshold tuning and test evaluation...")

In [ ]:
if not BASELINE_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Model file not found at {BASELINE_CHECKPOINT_PATH}. Run training first."
    )
best_model = training_utils.load_model_compat(BASELINE_CHECKPOINT_PATH)
print(f"Loaded model from: {BASELINE_CHECKPOINT_PATH}")
print(f"Model input shape: {best_model.input_shape}")
print(f"Model output shape: {best_model.output_shape}")

In [ ]:
# Threshold selection stage
print("Searching the best decision threshold on validation data (macro F1)...")

In [ ]:
# Threshold tuning on validation split (PNEUMONIA F1)
best_threshold, best_f1_val, _, _ = training_utils.tune_threshold(best_model, val_ds)
print(f"Best threshold (PNEUMONIA F1): {best_threshold:.2f}  |  Best F1: {best_f1_val:.4f}")

In [ ]:
# Evaluate on test set
baseline_metrics_dict, baseline_report, y_true, y_prob, y_pred = training_utils.evaluate_model(
    best_model, test_ds, best_threshold
)
test_roc_auc = baseline_metrics_dict["auc"]
print(f"Test ROC-AUC: {test_roc_auc:.4f}")

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=["NORMAL", "PNEUMONIA"],
    yticklabels=["NORMAL", "PNEUMONIA"],
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix (Test, threshold={best_threshold:.2f})")
plt.tight_layout()
plt.show()

print("Classification Report (Test):\n")
print(classification_report(y_true, y_pred, target_names=["NORMAL", "PNEUMONIA"], zero_division=0))

### 5.4 Save Model Artifacts

In [ ]:
# Persist baseline model artifacts and metadata
baseline_hyperparams = {
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "optimizer": "Adam",
    "learning_rate": 1e-4,
    "class_weight": class_weight,
    "balanced_sampling": "sample_from_datasets(weights=[0.5, 0.5])",
    "steps_per_epoch": steps_per_epoch,
}

if "history" not in globals():
    history = None

save_model_with_meta(
    best_model, "baseline_cnn", baseline_metrics_dict, history,
    baseline_hyperparams, best_threshold
)

print(f"Baseline CNN saved with threshold={best_threshold:.2f}")
print(f"  Accuracy : {baseline_metrics_dict['accuracy']:.4f}")
print(f"  AUC      : {baseline_metrics_dict['auc']:.4f}")
print(f"  Precision: {baseline_metrics_dict['precision']:.4f}")
print(f"  Recall   : {baseline_metrics_dict['recall']:.4f}")
print(f"  F1       : {baseline_metrics_dict['f1']:.4f}")

### 5.5 Probability Distribution Diagnostic

Inspect how predicted probabilities separate NORMAL vs PNEUMONIA on the test set.

In [ ]:
# Diagnostic: probability distribution by true class on test set
from IPython.display import display

# Flatten to 1D arrays to avoid shape/indexing edge cases.
y_true_arr = np.asarray(y_true).reshape(-1)
y_prob_arr = np.asarray(y_prob).reshape(-1)

normal_probs = y_prob_arr[y_true_arr == 0]
pneumonia_probs = y_prob_arr[y_true_arr == 1]

print(f"NORMAL samples: {normal_probs.size} | PNEUMONIA samples: {pneumonia_probs.size}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(normal_probs, bins=40, alpha=0.6, label="NORMAL", color="steelblue")
ax.hist(pneumonia_probs, bins=40, alpha=0.6, label="PNEUMONIA", color="salmon")
ax.axvline(best_threshold, color="black", linestyle="--", label=f"best_threshold={best_threshold:.2f}")
ax.axvline(0.5, color="gray", linestyle=":", label="0.5 (hardcoded)")
ax.set_xlabel("Predicted Probability (PNEUMONIA)")
ax.set_ylabel("Count")
ax.set_title("Test Set: Predicted Probability Distribution by True Class")
ax.legend()
fig.tight_layout()

display(fig)
plt.close(fig)

### 5.6 Sample Predictions

In [ ]:
# Show sample predictions from test set (balanced sample: 5 NORMAL + 4 PNEUMONIA)
normal_images, pneumonia_images = [], []
normal_true, pneumonia_true = [], []

for images, labels in test_ds:
    lbls = labels.numpy().ravel().astype(int)
    for i, lbl in enumerate(lbls):
        if lbl == 0 and len(normal_images) < 5:
            normal_images.append(images[i])
            normal_true.append(lbl)
        elif lbl == 1 and len(pneumonia_images) < 4:
            pneumonia_images.append(images[i])
            pneumonia_true.append(lbl)

    if len(normal_images) >= 5 and len(pneumonia_images) >= 4:
        break

sample_images = tf.stack(normal_images + pneumonia_images)
sample_true = np.array(normal_true + pneumonia_true, dtype=int)

sample_probs = best_model.predict(sample_images, verbose=0).ravel()
sample_preds = (sample_probs >= best_threshold).astype(int)

idx_to_class = {0: "NORMAL", 1: "PNEUMONIA"}

plt.figure(figsize=(12, 12))
plt.suptitle(f"Sample Test Predictions (threshold={best_threshold:.2f})", y=1.02)
for i in range(len(sample_images)):
    img = sample_images[i].numpy().astype("uint8")
    true_label = idx_to_class[int(sample_true[i])]
    pred_label = idx_to_class[int(sample_preds[i])]
    conf = sample_probs[i] if sample_preds[i] == 1 else (1 - sample_probs[i])

    plt.subplot(3, 3, i + 1)
    plt.imshow(img.astype("uint8"))
    plt.title(f"T:{true_label} | P:{pred_label}\nConf:{conf:.2f}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## 6. Overfitting Countermeasures

1. Medically safe data augmentation (`RandomFlip`, mild `RandomRotation`, mild `RandomTranslation`, mild `RandomZoom`, mild `RandomShear`, mild `RandomBrightness`).
2. Batch normalisation after each convolution block.
3. Progressive dropout in conv blocks plus classifier dropout.
4. L2 regularisation in convolution and dense layers.
5. Gradient clipping (`clipnorm=1.0`) in Adam optimiser.
6. Conservative optimiser settings (`learning_rate=0.001`, `label_smoothing=0.02`) for smoother convergence.
7. Early stopping and LR scheduling on `val_auc`.

## 7. Model C — Baseline CNN (Tuned Hyperparameters)

Use Bayesian optimisation (Keras Tuner) to search over architecture and training hyperparameters, then retrain the best configuration for the full epoch budget.

### 7.1 Tuner Configuration

In [ ]:
MAX_TRIALS = 15
TUNER_EPOCHS = 12
TUNER_SEED = 42


def _load_image_for_tuning(path, label):
    return data_utils.load_image(path, label, IMG_SIZE)

# Build once, then only re-batch per trial to avoid remapping/reloading images.
_train_ds_unbatched = (
    tf.data.Dataset.from_tensor_slices((train_paths, train_labels_raw.astype("float32")))
    .shuffle(len(train_paths), seed=TUNER_SEED, reshuffle_each_iteration=True)
    .map(_load_image_for_tuning, num_parallel_calls=tf.data.AUTOTUNE)
)
_val_ds_unbatched = (
    tf.data.Dataset.from_tensor_slices((val_paths, val_labels_arr.astype("float32")))
    .map(_load_image_for_tuning, num_parallel_calls=tf.data.AUTOTUNE)
)


def _rebatch_trial_datasets(batch_size):
    train_ds_trial = _train_ds_unbatched.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    val_ds_trial = _val_ds_unbatched.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return train_ds_trial, val_ds_trial


def build_tunable_model(hp):
    return model_utils.build_baseline_model_from_hp(
        hp,
        img_size=IMG_SIZE,
        augmentation_layer=data_augmentation,
    )


class PneumoniaTuner(kt.BayesianOptimization):
    def run_trial(self, trial, *args, **kwargs):
        hp = trial.hyperparameters
        batch_size = hp.Choice("batch_size", [16, 32, 48])

        train_ds_trial, val_ds_trial = _rebatch_trial_datasets(batch_size)
        trial_steps = max(1, int(np.ceil(len(train_paths) / batch_size)))

        trial_callbacks = [
            keras.callbacks.EarlyStopping(
                monitor="val_accuracy",
                mode="max",
                patience=3,
                restore_best_weights=True,
            )
        ]

        kwargs.update(
            {
                "x": train_ds_trial,
                "validation_data": val_ds_trial,
                "epochs": TUNER_EPOCHS,
                "steps_per_epoch": trial_steps,
                "callbacks": trial_callbacks,
                "class_weight": class_weight,
                "verbose": 1,
            }
        )
        return super().run_trial(trial, *args, **kwargs)

### 7.2 Run Hyperparameter Search

In [ ]:
tuned_callbacks = training_utils.get_training_callbacks(
    checkpoint_path=TUNED_CHECKPOINT_PATH,
    patience=5,
    monitor="val_auc",
)

tuner = PneumoniaTuner(
    hypermodel=build_tunable_model,
    objective=kt.Objective("val_accuracy", direction="max"),
    max_trials=MAX_TRIALS,
    directory=str(SAVE_DIR),
    project_name="baseline_cnn_tuning",
    overwrite=True,
)

tuner.search()
best_hp = tuner.get_best_hyperparameters(1)[0]
print("Best hyperparameters:", best_hp.values)
print("Best batch_size:", best_hp.get("batch_size"))

### 7.3 Train Best Model

In [ ]:
# Train (or resume) tuned baseline model using the best tuned batch size
best_batch_size = best_hp.get("batch_size") or 32
train_ds_best, val_ds_best = _rebatch_trial_datasets(best_batch_size)
best_steps_per_epoch = max(1, int(np.ceil(len(train_paths) / best_batch_size)))

if TUNED_CHECKPOINT_PATH.exists():
    print(f"Loading existing tuned checkpoint: {TUNED_CHECKPOINT_PATH}")
    baseline_tuned_model = training_utils.load_model_compat(TUNED_CHECKPOINT_PATH)
    baseline_tuned_history = None
else:
    baseline_tuned_model = model_utils.build_baseline_model_from_hp(
        best_hp,
        img_size=IMG_SIZE,
        augmentation_layer=data_augmentation,
    )
    baseline_tuned_history = baseline_tuned_model.fit(
        train_ds_best,
        validation_data=val_ds_best,
        epochs=EPOCHS,
        steps_per_epoch=best_steps_per_epoch,
        callbacks=tuned_callbacks,
        class_weight=class_weight,
        verbose=1,
    )
    print(f"Best tuned checkpoint saved to: {TUNED_CHECKPOINT_PATH}")

### 7.4 Threshold Tuning & Test Evaluation

In [ ]:
# Threshold tuning (PNEUMONIA F1) for tuned model
baseline_tuned_threshold, baseline_tuned_f1_val, _, _ = training_utils.tune_threshold(
    baseline_tuned_model, val_ds
)
print(f"Tuned model threshold (PNEUMONIA F1): {baseline_tuned_threshold:.2f}  |  Val F1: {baseline_tuned_f1_val:.4f}")

# Evaluate on test set
baseline_tuned_metrics, tuned_report, y_true, y_prob, y_pred = training_utils.evaluate_model(
    baseline_tuned_model, test_ds, baseline_tuned_threshold
)
print(f"Tuned model test ROC-AUC: {baseline_tuned_metrics['auc']:.4f}")

### 7.5 Save Model Artifacts

In [ ]:
# Save tuned model and metadata
save_model_with_meta(
    baseline_tuned_model,
    "baseline_cnn_tuned",
    baseline_tuned_metrics,
    baseline_tuned_history,
    best_hp.values,
    baseline_tuned_threshold,
)
print(f"Saved tuned baseline model with threshold {baseline_tuned_threshold:.2f}")

### 7.6 Training Curves

In [ ]:
# Plot training curves (loss, accuracy, and AUC)
if baseline_tuned_history is None:
    print("Training history is unavailable because model was loaded from checkpoint.")
else:
    plt.figure(figsize=(15, 4))

    plt.subplot(1, 3, 1)
    plt.plot(baseline_tuned_history.history["accuracy"], label="Train Accuracy")
    plt.plot(baseline_tuned_history.history["val_accuracy"], label="Val Accuracy")
    plt.title("Accuracy Over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()

    plt.subplot(1, 3, 2)
    plt.plot(baseline_tuned_history.history["loss"], label="Train Loss")
    plt.plot(baseline_tuned_history.history["val_loss"], label="Val Loss")
    plt.title("Loss Over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    plt.subplot(1, 3, 3)
    plt.plot(baseline_tuned_history.history["auc"], label="Train AUC")
    plt.plot(baseline_tuned_history.history["val_auc"], label="Val AUC")
    plt.title("AUC Over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("AUC")
    plt.legend()

    plt.tight_layout()
    plt.show()

### 7.7 Confusion Matrix & Classification Report

In [ ]:
# Test set confusion matrix for tuned model
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=["NORMAL", "PNEUMONIA"],
    yticklabels=["NORMAL", "PNEUMONIA"],
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix – Tuned (threshold={baseline_tuned_threshold:.2f})")
plt.tight_layout()
plt.show()

print("Classification Report (Test):\n")
print(classification_report(y_true, y_pred, target_names=["NORMAL", "PNEUMONIA"], zero_division=0))

### 7.8 Sample Predictions

In [ ]:
# Show sample predictions from test set (balanced sample: 5 NORMAL + 4 PNEUMONIA)
normal_images, pneumonia_images = [], []
normal_true, pneumonia_true = [], []

for images, labels in test_ds:
    lbls = labels.numpy().ravel().astype(int)
    for i, lbl in enumerate(lbls):
        if lbl == 0 and len(normal_images) < 5:
            normal_images.append(images[i])
            normal_true.append(lbl)
        elif lbl == 1 and len(pneumonia_images) < 4:
            pneumonia_images.append(images[i])
            pneumonia_true.append(lbl)

    if len(normal_images) >= 5 and len(pneumonia_images) >= 4:
        break

sample_images = tf.stack(normal_images + pneumonia_images)
sample_true = np.array(normal_true + pneumonia_true, dtype=int)

sample_probs = baseline_tuned_model.predict(sample_images, verbose=0).ravel()
sample_preds = (sample_probs >= best_threshold).astype(int)

idx_to_class = {0: "NORMAL", 1: "PNEUMONIA"}

plt.figure(figsize=(12, 12))
plt.suptitle(f"Sample Test Predictions (threshold={best_threshold:.2f})", y=1.02)
for i in range(len(sample_images)):
    img = sample_images[i].numpy().astype("uint8")
    true_label = idx_to_class[int(sample_true[i])]
    pred_label = idx_to_class[int(sample_preds[i])]
    conf = sample_probs[i] if sample_preds[i] == 1 else (1 - sample_probs[i])

    plt.subplot(3, 3, i + 1)
    plt.imshow(img.astype("uint8"))
    plt.title(f"T:{true_label} | P:{pred_label}\nConf:{conf:.2f}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## 8. Conclusion — CNN Model Comparison

Load saved metadata for all three models and present a side-by-side comparison.

In [ ]:
# ── Final comparison: all three CNN models ──────────────────────────────────

records = []
for model_name, label in [
    ("baseline_cnn_no_aug",  "Model A – No Augmentation"),
    ("baseline_cnn",         "Model B – With Augmentation"),
    ("baseline_cnn_tuned",   "Model C – Tuned HP"),
]:
    meta = training_utils.load_model_meta(SAVE_DIR, model_name)
    if meta is None:
        print(f"  {label}: metadata not found, skipping")
        continue
    m = meta["metrics"]
    records.append({
        "Model": label,
        "Accuracy": round(m.get("accuracy", 0), 4),
        "AUC": round(m.get("auc", 0), 4),
        "Precision": round(m.get("precision", 0), 4),
        "Recall": round(m.get("recall", 0), 4),
        "F1": round(m.get("f1", 0), 4),
        "Threshold": round(meta.get("threshold", 0), 2),
    })

if records:
    df = pd.DataFrame(records).set_index("Model")
    print(df.to_string())
    print()

    # Highlight best per metric
    for col in ["Accuracy", "AUC", "Precision", "Recall", "F1"]:
        best_model = df[col].idxmax()
        print(f"  Best {col}: {best_model} ({df.loc[best_model, col]:.4f})")

    # Bar chart
    ax = df[["AUC", "Recall", "Precision", "F1"]].plot(
        kind="bar", figsize=(10, 5), colormap="tab10", edgecolor="white",
    )
    ax.set_ylim(0.5, 1.02)
    ax.set_ylabel("Score")
    ax.set_title("CNN Model Comparison — Test Set Metrics")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha="right")
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()
else:
    print("No model metadata found. Run training cells first.")

### Key Takeaways

- **Data augmentation** (Model B vs A) is expected to improve generalisation by exposing the model to plausible image variations and reducing overfitting to the limited training set.
- **Hyperparameter tuning** (Model C) systematically searches over architecture depth, filter counts, dropout rates, L2 strength, and learning rate to find a better configuration than the hand-tuned baseline.
- **Balanced class weighting** (via `compute_class_weights()`) is applied consistently across all three models to compensate for the PNEUMONIA-heavy class imbalance, giving more weight to the minority NORMAL class.
- **Decision threshold tuning** on the validation set (optimising PNEUMONIA F1) further improves the recall–precision trade-off beyond the default 0.5 cutoff.

The best-performing CNN model from this notebook is carried forward as the baseline reference for the **transfer learning experiments** in notebook **03**.